# Model Comparison — Telecom Customer Churn

This notebook establishes the model-comparison workflow.

Objectives:
- Reuse the production data-preparation code in src/
- Reuse the stratified train/test split
- Reuse the leakage-safe preprocessing pipeline
- Evaluate candidate models using stratified out-of-fold predictions
- Evaluate the final candidate on the held-out test set
- Verify that the comparison framework reproduces the existing
  Logistic Regression baseline before introducing additional models

In [3]:
from pathlib import Path
import sys
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.preprocessing import (
    build_preprocessor,
    load_raw_data,
    prepare_dataset,
)
from src.data.split import create_train_test_split
from src.modeling.comparison import (
    evaluate_model_cv,
    fit_and_evaluate_test,
)

In [4]:
df = load_raw_data()

X, y = prepare_dataset(df)

X_train, X_test, y_train, y_test = create_train_test_split(
    X,
    y,
)

print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
print(f"Training churn rate: {y_train.mean():.6f}")
print(f"Test churn rate: {y_test.mean():.6f}")

Training rows: 6762
Test rows: 1691
Training churn rate: 0.064922
Test churn rate: 0.065050


In [5]:
logistic_regression = Pipeline(
    steps=[
        ('preprocessor', build_preprocessor(scale_numeric=True)),
        ('classifier', LogisticRegression(max_iter=1000, random_state=42))
    ])

In [6]:
logistic_cv_result = evaluate_model_cv(
    model=logistic_regression,
    X=X_train,
    y=y_train
)

logistic_cv_result["metrics"]

{'PR-AUC': 0.08264539890467607, 'ROC-AUC': 0.5671707981527467}

In [7]:
logistic_test_result = fit_and_evaluate_test(
    model=logistic_regression,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    threshold=0.06
)

logistic_test_result["metrics"]

Threshold              0.060000
PR-AUC                 0.082143
ROC-AUC                0.559192
Precision              0.070406
Recall                 0.536364
F1                     0.124473
Customers Flagged    838.000000
Coverage               0.495565
dtype: float64

In [9]:
comparison = pd.DataFrame(
    {
        "Logistic Regression": logistic_test_result["metrics"]
    }
).T

comparison

,Threshold,PR-AUC,ROC-AUC,Precision,Recall,F1,Customers Flagged,Coverage
Logistic Regression,0.06,0.082143,0.559192,0.070406,0.536364,0.124473,838.0,0.495565


In [10]:
from sklearn.ensemble import RandomForestClassifier

In [12]:
random_forest = Pipeline(
    steps=[
        ('preprocessor', build_preprocessor(scale_numeric=False)),
        ('classifier', RandomForestClassifier(
                n_estimators=300,
                max_depth=None,
                min_samples_leaf=5,
                class_weight='balanced',
                random_state=42,
                n_jobs=-1
            ))
    ]
)

In [13]:
random_forest_cv_result = evaluate_model_cv(
    model=random_forest,
    X=X_train,
    y=y_train,
)

random_forest_cv_result["metrics"]

{'PR-AUC': 0.0806682669996597, 'ROC-AUC': 0.5618054922604211}

In [14]:
random_forest_test_result = fit_and_evaluate_test(
    model=random_forest,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    threshold=0.06,
)

random_forest_test_result["metrics"]

Threshold               0.060000
PR-AUC                  0.083352
ROC-AUC                 0.566592
Precision               0.065050
Recall                  1.000000
F1                      0.122154
Customers Flagged    1691.000000
Coverage                1.000000
dtype: float64

In [15]:
comparison = pd.DataFrame(
    {
        "Logistic Regression": logistic_test_result["metrics"],
        "Random Forest": random_forest_test_result["metrics"],
    }
).T

comparison

,Threshold,PR-AUC,ROC-AUC,Precision,Recall,F1,Customers Flagged,Coverage
Logistic Regression,0.06,0.082143,0.559192,0.070406,0.536364,0.124473,838.0,0.495565
Random Forest,0.06,0.083352,0.566592,0.065050,1.000000,0.122154,1691.0,1.000000


In [16]:
rf_proba = random_forest_test_result["test_probabilities"]

pd.Series(rf_proba).describe(
    percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

count    1691.000000
mean        0.395386
std         0.118971
min         0.123477
1%          0.175563
5%          0.217697
10%         0.248615
25%         0.302554
50%         0.386784
75%         0.475490
90%         0.554441
95%         0.605230
99%         0.697157
max         0.802524
dtype: float64

In [17]:
print(f"Minimum probability: {rf_proba.min():.6f}")
print(f"Maximum probability: {rf_proba.max():.6f}")
print(f"Probability >= 0.06: {(rf_proba >= 0.06).mean():.6f}")

Minimum probability: 0.123477
Maximum probability: 0.802524
Probability >= 0.06: 1.000000


In [18]:
comparison["Threshold"] = comparison["Threshold"].astype(float)
comparison

,Threshold,PR-AUC,ROC-AUC,Precision,Recall,F1,Customers Flagged,Coverage
Logistic Regression,0.06,0.082143,0.559192,0.070406,0.536364,0.124473,838.0,0.495565
Random Forest,0.06,0.083352,0.566592,0.065050,1.000000,0.122154,1691.0,1.000000


> The 0.06 threshold was established during Logistic Regression
> validation. It is shown here for reference and is not assumed to
> be an appropriate production threshold for Random Forest.
> Model ranking is therefore based primarily on threshold-independent
> probability metrics such as PR-AUC and ROC-AUC at this stage.

In [19]:
from src.modeling.thresholds import evaluate_thresholds

rf_oof_proba = random_forest_cv_result["oof_predictions"]

rf_threshold_results = evaluate_thresholds(
    y_true=y_train,
    y_proba=rf_oof_proba,
    thresholds=[
        0.05,
        0.10,
        0.15,
        0.20,
        0.25,
        0.30,
        0.35,
        0.40,
        0.45,
        0.50,
        0.55,
        0.60,
    ],
)

rf_threshold_results.sort_values(
    "F1",
    ascending=False,
)

,Threshold,Customers Flagged,Coverage,Precision,Recall,F1
6,0.35,3949,0.583999,0.077235,0.694761,0.139015
7,0.40,2988,0.441881,0.077309,0.526196,0.134812
8,0.45,2078,0.307306,0.079403,0.375854,0.131108
9,0.50,1291,0.190920,0.085980,0.252847,0.128324
5,0.30,5021,0.742532,0.069508,0.794989,0.127839
4,0.25,6112,0.903875,0.065772,0.915718,0.122729
3,0.20,6602,0.976338,0.065283,0.981777,0.122426
0,0.05,6762,1.000000,0.064922,1.000000,0.121928
1,0.10,6762,1.000000,0.064922,1.000000,0.121928
2,0.15,6746,0.997634,0.064927,0.997722,0.121921


In [20]:
rf_test_at_035 = fit_and_evaluate_test(
    model=random_forest,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    threshold=0.35,
)

rf_test_at_035["metrics"]

Threshold               0.350000
PR-AUC                  0.083352
ROC-AUC                 0.566592
Precision               0.076550
Recall                  0.718182
F1                      0.138354
Customers Flagged    1032.000000
Coverage                0.610290
dtype: float64

In [21]:
comparison_thresholded = pd.DataFrame(
    {
        "Logistic Regression": logistic_test_result["metrics"],
        "Random Forest": rf_test_at_035["metrics"],
    }
).T

comparison_thresholded

,Threshold,PR-AUC,ROC-AUC,Precision,Recall,F1,Customers Flagged,Coverage
Logistic Regression,0.06,0.082143,0.559192,0.070406,0.536364,0.124473,838.0,0.495565
Random Forest,0.35,0.083352,0.566592,0.076550,0.718182,0.138354,1032.0,0.610290
